# 01 · Understand the source snapshot

Research question: what does each row represent, and which questions can this snapshot answer? Inspect all seven sources before adopting a fact table. Raw hashes protect the supplied files.

In [1]:
from pathlib import Path
import sys
import io
import pandas as pd
import numpy as np
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_loader import load_raw, profile_sources, verify_raw, TABLES
from src.pipeline import prepare
from src.analysis import *
from src.visualization import *
def show(figure):
    buffer = io.BytesIO()
    figure.savefig(buffer, format='png', bbox_inches='tight')
    display(Image(data=buffer.getvalue()))
    plt.close(figure)

## Load and verify

All sources are local. No Kaggle credentials or network download are needed.

In [2]:
print(f"Verified {verify_raw()} original source hashes")
raw = load_raw()
profiles = profile_sources(raw)
display(profiles['source_summary'])
display(profiles['date_coverage'])

Verified 7 original source hashes


,table,rows,columns,duplicate_rows,duplicate_keys,missing_keys
0,event,1259,4,0,0,0
1,fight,11441,14,0,0,0
2,fighter,4581,16,0,0,0
3,fighter_bonus,2409,2,0,0,0
4,master,11441,93,0,0,0
5,round,25131,48,0,0,0
6,scrape_error,330,9,0,0,0


,table,column,minimum,maximum,malformed
0,event,date,1993-11-12 00:00:00,2026-08-08 00:00:00,0
1,fighter,dob,1943-01-25 00:00:00,2005-12-20 00:00:00,0
2,master,event_date,1993-11-12 00:00:00,2026-08-08 00:00:00,0
3,scrape_error,occurred_at,2026-08-11 05:57:43,2026-08-11 06:39:59,0


## Inspect event.csv

Inspect sample rows and every column’s inferred type, missingness, uniqueness and example values.

In [3]:
display(raw['event'].head(3))
with pd.option_context('display.max_rows', 110, 'display.max_colwidth', 70):
    display(profiles['data_dictionary'].query("table == 'event'"))

,event_id,event_name,date,location
0,495add4fbede0a44,UFC Fight Night: Gamrot vs. Salkilld,2026-08-08,"Las Vegas, Nevada, USA"
1,1a50e734bb54861a,UFC Fight Night: Medic vs. Rodriguez,2026-08-01,"Belgrade, Vojvodina, Serbia"
2,681d07e328798ec0,UFC Fight Night: Ankalaev vs. Guskov,2026-07-25,"Abu Dhabi, Abu Dhabi, United Arab Emirates"


,table,column,dtype,missing,missing_pct,unique,examples
0,event,event_id,string,0,0.000000,1259,495add4fbede0a44 | 1a50e734bb54861a | 681d07e328798ec0
1,event,event_name,str,0,0.000000,1259,UFC Fight Night: Gamrot vs. Salkilld | UFC Fight Night: Medic vs. ...
2,event,date,str,0,0.000000,1214,2026-08-08 | 2026-08-01 | 2026-07-25
3,event,location,str,476,37.807784,174,"Las Vegas, Nevada, USA | Belgrade, Vojvodina, Serbia | Abu Dhabi, ..."


## Inspect fight.csv

Inspect sample rows and every column’s inferred type, missingness, uniqueness and example values.

In [4]:
display(raw['fight'].head(3))
with pd.option_context('display.max_rows', 110, 'display.max_colwidth', 70):
    display(profiles['data_dictionary'].query("table == 'fight'"))

,event_id,fight_id,weight_class,title_fight,r_id,b_id,winner_id,result_status,method,round,time,time_format,referee,details
0,baf942f4bcb09894,4cd909fa0d3e0483,Welterweight,0,c3ea8deeba00462b,ad32471f01e7b1a5,c3ea8deeba00462b,win,Submission,2,1:11,3 Rnd (5-5-5),Herb Dean,Injury \n Neck Injury
1,c61f66d8c3fd5f07,5ac9cf4ffc15ccca,Welterweight,0,f57a8a52ca401ed9,ad32471f01e7b1a5,f57a8a52ca401ed9,win,Submission,3,2:04,3 Rnd (5-5-5),Steve Mazzagatti,Armbar From Bottom Guard
2,80e6d71e14c1d356,e8ec95cdef577733,Welterweight,0,79899ecf62020f6d,ad32471f01e7b1a5,79899ecf62020f6d,win,KO/TKO,1,1:27,3 Rnd (5-5-5),NaN,Punches to Head From Mount


,table,column,dtype,missing,missing_pct,unique,examples
4,fight,event_id,string,0,0.000000,1259,baf942f4bcb09894 | c61f66d8c3fd5f07 | 80e6d71e14c1d356
5,fight,fight_id,string,0,0.000000,11441,4cd909fa0d3e0483 | 5ac9cf4ffc15ccca | e8ec95cdef577733
6,fight,weight_class,str,0,0.000000,154,Welterweight | Middleweight | Light Heavyweight
7,fight,title_fight,int64,0,0.000000,2,0 | 1
8,fight,r_id,string,0,0.000000,2713,c3ea8deeba00462b | f57a8a52ca401ed9 | 79899ecf62020f6d
9,fight,b_id,string,0,0.000000,3650,ad32471f01e7b1a5 | 13b2f59210dda9cc | 253de74457bd6374
10,fight,winner_id,string,203,1.774320,2517,c3ea8deeba00462b | f57a8a52ca401ed9 | 79899ecf62020f6d
11,fight,result_status,str,0,0.000000,3,win | no_contest | draw
12,fight,method,str,0,0.000000,11,Submission | KO/TKO | DQ
13,fight,round,int64,0,0.000000,6,2 | 3 | 1


## Inspect fighter.csv

Inspect sample rows and every column’s inferred type, missingness, uniqueness and example values.

In [5]:
display(raw['fighter'].head(3))
with pd.option_context('display.max_rows', 110, 'display.max_colwidth', 70):
    display(profiles['data_dictionary'].query("table == 'fighter'"))

,fighter_id,fighter_name,fighter_nick_name,height,weight_lbs,reach_inches,stance,dob,slpm,str_acc,sapm,str_def,td_avg,td_acc,td_def,sub_avg
0,ad32471f01e7b1a5,Jeremy Jackson,Scorpion,"5' 10""",170.0,NaN,Orthodox,1982-09-19,0.71,27,2.03,25,1.52,40,58,0.0
1,13b2f59210dda9cc,Kevin Jackson,NaN,"5' 10""",199.0,NaN,Switch,1964-11-25,0.00,0,0.00,0,0.00,0,0,0.0
2,b4eeac608fc82f88,Yves Jabouin,Tiger,"5' 7""",135.0,68.0,Orthodox,1979-05-30,3.06,41,2.93,62,1.94,60,82,0.1


,table,column,dtype,missing,missing_pct,unique,examples
18,fighter,fighter_id,string,0,0.000000,4581,ad32471f01e7b1a5 | 13b2f59210dda9cc | b4eeac608fc82f88
19,fighter,fighter_name,str,0,0.000000,4573,Jeremy Jackson | Kevin Jackson | Yves Jabouin
20,fighter,fighter_nick_name,str,2029,44.291639,1998,Scorpion | Tiger | Rampage
21,fighter,height,str,372,8.120498,27,"5' 10"" | 5' 7"" | 6' 1"""
22,fighter,weight_lbs,float64,88,1.920978,113,170.0 | 199.0 | 135.0
23,fighter,reach_inches,float64,1992,43.483955,27,68.0 | 73.0 | 71.0
24,fighter,stance,str,897,19.580878,5,Orthodox | Switch | Southpaw
25,fighter,dob,str,510,11.132940,3405,1982-09-19 | 1964-11-25 | 1979-05-30
26,fighter,slpm,float64,0,0.000000,716,0.71 | 0.0 | 3.06
27,fighter,str_acc,int64,0,0.000000,83,27 | 0 | 41


## Inspect fighter_bonus.csv

Inspect sample rows and every column’s inferred type, missingness, uniqueness and example values.

In [6]:
display(raw['fighter_bonus'].head(3))
with pd.option_context('display.max_rows', 110, 'display.max_colwidth', 70):
    display(profiles['data_dictionary'].query("table == 'fighter_bonus'"))

,fight_id,bonus_type
0,21392a61a2cf24a3,Knockout of the Night
1,1bf7f647f309464e,Performance of the Night
2,8348ff428d1687ab,Submission of the Night


,table,column,dtype,missing,missing_pct,unique,examples
34,fighter_bonus,fight_id,string,0,0.0,2351,21392a61a2cf24a3 | 1bf7f647f309464e | 8348ff428d1687ab
35,fighter_bonus,bonus_type,str,0,0.0,4,Knockout of the Night | Performance of the Night | Submission of t...


## Inspect master.csv

Inspect sample rows and every column’s inferred type, missingness, uniqueness and example values.

In [7]:
display(raw['master'].head(3))
with pd.option_context('display.max_rows', 110, 'display.max_colwidth', 70):
    display(profiles['data_dictionary'].query("table == 'master'"))

,fight_id,event_id,event_name,event_date,event_location,weight_class,title_fight,winner_id,result_status,method,...,r_total_sig_str_atmp_clinch,b_total_sig_str_landed_clinch,b_total_sig_str_atmp_clinch,r_total_sig_str_landed_ground,r_total_sig_str_atmp_ground,b_total_sig_str_landed_ground,b_total_sig_str_atmp_ground,r_total_ctrl_seconds,b_total_ctrl_seconds,rounds_fought
0,4cd909fa0d3e0483,baf942f4bcb09894,The Ultimate Fighter: The Comeback Finale,2006-11-11,"Las Vegas, Nevada, USA",Welterweight,0,c3ea8deeba00462b,win,Submission,...,3,5,5,5,7,0,1,67,201,2
1,5ac9cf4ffc15ccca,c61f66d8c3fd5f07,UFC 44: Undisputed,2003-09-26,"Las Vegas, Nevada, USA",Welterweight,0,f57a8a52ca401ed9,win,Submission,...,0,1,5,16,21,8,25,402,208,3
2,e8ec95cdef577733,80e6d71e14c1d356,KOTC 15: Bad Intentions,2002-06-22,NaN,Welterweight,0,79899ecf62020f6d,win,KO/TKO,...,0,0,0,11,15,0,0,0,0,1


,table,column,dtype,missing,missing_pct,unique,examples
36,master,fight_id,string,0,0.000000,11441,4cd909fa0d3e0483 | 5ac9cf4ffc15ccca | e8ec95cdef577733
37,master,event_id,string,0,0.000000,1259,baf942f4bcb09894 | c61f66d8c3fd5f07 | 80e6d71e14c1d356
38,master,event_name,str,0,0.000000,1259,The Ultimate Fighter: The Comeback Finale | UFC 44: Undisputed | K...
39,master,event_date,str,0,0.000000,1214,2006-11-11 | 2003-09-26 | 2002-06-22
40,master,event_location,str,2621,22.908837,174,"Las Vegas, Nevada, USA | New Orleans, Louisiana, USA | Yokohama, K..."
41,master,weight_class,str,0,0.000000,154,Welterweight | Middleweight | Light Heavyweight
42,master,title_fight,int64,0,0.000000,2,0 | 1
43,master,winner_id,string,203,1.774320,2517,c3ea8deeba00462b | f57a8a52ca401ed9 | 79899ecf62020f6d
44,master,result_status,str,0,0.000000,3,win | no_contest | draw
45,master,method,str,0,0.000000,11,Submission | KO/TKO | DQ


## Inspect round.csv

Inspect sample rows and every column’s inferred type, missingness, uniqueness and example values.

In [8]:
display(raw['round'].head(3))
with pd.option_context('display.max_rows', 110, 'display.max_colwidth', 70):
    display(profiles['data_dictionary'].query("table == 'round'"))

,fight_id,round_no,r_id,b_id,r_kd,b_kd,r_sig_landed,r_sig_atmp,b_sig_landed,b_sig_atmp,...,b_sig_str_landed_distance,b_sig_str_atmp_distance,r_sig_str_landed_clinch,r_sig_str_atmp_clinch,b_sig_str_landed_clinch,b_sig_str_atmp_clinch,r_sig_str_landed_ground,r_sig_str_atmp_ground,b_sig_str_landed_ground,b_sig_str_atmp_ground
0,4cd909fa0d3e0483,1,c3ea8deeba00462b,ad32471f01e7b1a5,0,0,11,14,5,12,...,0,6,3,3,5,5,5,7,0,1
1,4cd909fa0d3e0483,2,c3ea8deeba00462b,ad32471f01e7b1a5,0,0,1,1,0,2,...,0,2,0,0,0,0,0,0,0,0
2,5ac9cf4ffc15ccca,1,f57a8a52ca401ed9,ad32471f01e7b1a5,0,0,8,12,3,7,...,0,2,0,0,1,3,8,12,2,2


,table,column,dtype,missing,missing_pct,unique,examples
129,round,fight_id,string,0,0.000000,11111,4cd909fa0d3e0483 | 5ac9cf4ffc15ccca | e8ec95cdef577733
130,round,round_no,int64,0,0.000000,6,1 | 2 | 3
131,round,r_id,string,0,0.000000,2544,c3ea8deeba00462b | f57a8a52ca401ed9 | 79899ecf62020f6d
132,round,b_id,string,0,0.000000,3455,ad32471f01e7b1a5 | 13b2f59210dda9cc | 253de74457bd6374
133,round,r_kd,int64,0,0.000000,5,0 | 1 | 2
134,round,b_kd,int64,0,0.000000,6,0 | 1 | 2
135,round,r_sig_landed,int64,0,0.000000,96,11 | 1 | 8
136,round,r_sig_atmp,int64,0,0.000000,171,14 | 1 | 12
137,round,b_sig_landed,int64,0,0.000000,93,5 | 0 | 3
138,round,b_sig_atmp,int64,0,0.000000,165,12 | 2 | 7


## Inspect scrape_error.csv

Inspect sample rows and every column’s inferred type, missingness, uniqueness and example values.

In [9]:
display(raw['scrape_error'].head(3))
with pd.option_context('display.max_rows', 110, 'display.max_colwidth', 70):
    display(profiles['data_dictionary'].query("table == 'scrape_error'"))

,error_id,entity_type,entity_id,context,url,error_name,description,occurred_at,resolved
0,1,fight,fc89b69a4fe79603,NaN,http://ufcstats.com/fight-details/fc89b69a4fe7...,PartialFightParse,"expected 2 round-by-round tables, found 0 -- t...",2026-08-11 05:57:43,0
1,2,fight,9325bcf98fa2a15e,NaN,http://ufcstats.com/fight-details/9325bcf98fa2...,PartialFightParse,"expected 2 round-by-round tables, found 0 -- t...",2026-08-11 05:57:49,0
2,3,fight,6035a68279b69aeb,NaN,http://ufcstats.com/fight-details/6035a68279b6...,PartialFightParse,"expected 2 round-by-round tables, found 0 -- t...",2026-08-11 05:57:52,0


,table,column,dtype,missing,missing_pct,unique,examples
177,scrape_error,error_id,string,0,0.0,330,1 | 2 | 3
178,scrape_error,entity_type,str,0,0.0,1,fight
179,scrape_error,entity_id,string,0,0.0,330,fc89b69a4fe79603 | 9325bcf98fa2a15e | 6035a68279b69aeb
180,scrape_error,context,float64,330,100.0,0,
181,scrape_error,url,str,0,0.0,330,http://ufcstats.com/fight-details/fc89b69a4fe79603 | http://ufcsta...
182,scrape_error,error_name,str,0,0.0,1,PartialFightParse
183,scrape_error,description,str,0,0.0,1,"expected 2 round-by-round tables, found 0 -- this fight likely has..."
184,scrape_error,occurred_at,str,0,0.0,265,2026-08-11 05:57:43 | 2026-08-11 05:57:49 | 2026-08-11 05:57:52
185,scrape_error,resolved,int64,0,0.0,1,0


## Relationships and categorical coverage

event → fight is one-to-many; fighter joins twice to fight; round has a fight + round key; bonus has a fight + category key. Names are not keys.

In [10]:
for source, column, target, key in [('fight','event_id','event','event_id'),
                                   ('fight','r_id','fighter','fighter_id'),
                                   ('fight','b_id','fighter','fighter_id'),
                                   ('round','fight_id','fight','fight_id'),
                                   ('fighter_bonus','fight_id','fight','fight_id')]:
    print(source, column, 'orphan references:', (~raw[source][column].isin(raw[target][key])).sum())
display(profiles['category_counts'].query("column in ['result_status','method','stance','bonus_type','error_name']"))

fight event_id orphan references: 0
fight r_id orphan references: 0
fight b_id orphan references: 0
round fight_id orphan references: 0


fighter_bonus fight_id orphan references: 0


,table,column,value,count
331,fight,result_status,win,11238
332,fight,result_status,no_contest,118
333,fight,result_status,draw,85
334,fight,method,Decision - Unanimous,3860
335,fight,method,KO/TKO,3715
336,fight,method,Submission,2411
337,fight,method,Decision - Split,1015
338,fight,method,TKO - Doctor's Stoppage,152
339,fight,method,Decision - Majority,117
340,fight,method,Overturned,71


## Why master is insufficient

The wide master has the same fight IDs, joined attributes and aggregate statistics. It loses round trajectories, omits fighters without a fight and zero-fills missing round totals. Lifetime profile metrics also contain future information for historical fights.

In [11]:
missing_round_ids = set(raw['fight'].fight_id) - set(raw['round'].fight_id)
affected = raw['master'][raw['master'].fight_id.isin(missing_round_ids)]
display(affected[['fight_id','event_date','r_total_sig_landed','b_total_sig_landed','rounds_fought']].head(10))
print('Fights with no round records:', len(affected))
print('Logged fight errors:', raw['scrape_error'].entity_id.nunique())
print('Profiles without observed fights:', (~raw['fighter'].fighter_id.isin(set(raw['fight'].r_id) | set(raw['fight'].b_id))).sum())

,fight_id,event_date,r_total_sig_landed,b_total_sig_landed,rounds_fought
95,fc89b69a4fe79603,2008-06-27,0,0,0
125,9325bcf98fa2a15e,2011-08-12,0,0,0
130,6035a68279b69aeb,2011-01-07,0,0,0
197,7ffcc3a72e082ace,1995-12-16,0,0,0
275,93908423a3e7c251,2001-11-03,0,0,0
333,89229d26e5a75a32,2008-11-21,0,0,0
375,a3355e0d592bdbbc,2007-02-10,0,0,0
401,ee84a0b122cd217a,2008-06-27,0,0,0
482,4d7231ec867553d3,2008-02-01,0,0,0
484,670915f96aa81e03,2011-08-12,0,0,0


Fights with no round records: 330
Logged fight errors: 330
Profiles without observed fights: 457


## Research extension: Where can the dataset mislead us?

The following answers are computed from the current snapshot. Tables expose denominators and missingness so the claims can be checked.

In [12]:
from src.research import run_chapter, display_answers
deep_tables, deep_answers = run_chapter('01')
display_answers(deep_answers)

### How much would using the complete master change the UFC population?

**Answer.** It would add 2,621 fights (22.9% of all source fights) beyond the 8,820-fight reporting cohort.

**Interpretation.** This is a scope mismatch, not additional UFC coverage. Non-UFC results must not enter UFC counts or prior-UFC records.

### Are missing round statistics a random subset of UFC fights?

**Answer.** 21 scoped fights lack statistics; their mean event year is 1996.0.

**Interpretation.** Coverage differs historically. Modern activity cannot be fairly compared with early raw totals without displaying observation rates.

### How much does zero-filling missing fights change average striking totals?

**Answer.** The source master gives 73.89 combined significant strikes per scoped fight; the complete observed subset gives 74.06.

**Interpretation.** The aggregate difference is small because scoped missingness is rare, but early-period comparisons can be affected much more. Missing control is a separate coverage problem.

In [13]:
display(deep_tables['deep_scope_bias'])

,scope,fights,events,finish_rate,complete_stats
0,Other promotion / feeder,2613,475,0.637199,0.881362
1,UFC 1994-2026,8820,783,0.521882,0.997619
2,UFC outside reporting window,8,1,1.000000,1.000000


In [14]:
display(deep_tables['deep_missingness'])

,period,appearances,age_missing,reach_missing,control_missing,statistics_missing
0,1994-1999,448,0.247768,0.892857,0.901786,0.09375
1,2000-2009,2082,0.013449,0.232949,0.000000,0.00000
2,2010-2019,8392,0.000596,0.051954,0.000000,0.00000
3,2020-2026,6718,0.000000,0.000000,0.000000,0.00000


In [15]:
display(deep_tables['deep_missing_selection'])

,stats_observed,fights,finish_rate,mean_year,duration_minutes
0,False,21,0.904762,1996.000000,3.950000
1,True,8799,0.520968,2016.568587,10.606739


In [16]:
display(deep_tables['deep_zero_bias'])

,scope,all_fights,observed_fights,naive_master_mean,observed_mean,missing_fights
0,Selected UFC fights,8820,8799,73.888209,74.064553,21
